# Verify Dataset Quality Checks

This notebook uses pandas to independently verify quality-check results for the taxi trips, taxi zones, weather, and air quality datasets. Change `DATASET` in the first code cell to select the dataset.

In [1]:
from pathlib import Path
import pandas as pd

# Set this to: taxi_trips, taxi_zones, weather, or air_quality.
DATASET = "taxi_trips"

# Windows workspace path. In Docker, use Path("/opt/spark/data").
DATA_ROOT = Path(r"/Users/dimitris/Documents/KTH/Year1/P1/Data-Intensive Computing ID2221/Assignment1/Data-Ingestion-Framework/landing")
# DATA_ROOT = Path("/opt/spark/data")

DATASETS = {
    "taxi_trips": {
        "directory": "taxi",
        "files": [
            "yellow_tripdata_2024-01.parquet",
            "yellow_tripdata_2024-02.parquet",
            "yellow_tripdata_2024-03.parquet",
        ],
        "format": "parquet",
    },
    "taxi_zones": {
        "directory": "taxi_zone_lookup",
        "files": ["taxi_zone_lookup.csv"],
        "format": "csv",
    },
    "weather": {
        "directory": "weather",
        "files": ["weather.csv"],
        "format": "csv",
    },
    "air_quality": {
        "directory": "air_quality",
        "files": ["hourly_88101_2024.csv"],
        "format": "csv",
    },
}

# Edit this section to verify a different rule set or expected Spark counts.
QUALITY_RULES = {
    "taxi_trips": [
        {"action": "validate_null_values", "column": "*"},
        {"action": "validate_duplicates", "columns": ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_fwd_flag', 'pickup_zone_id', 'dropoff_zone_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']},
        {"action": "validate_allowed_values", "column": "vendor_id", "allowed_values": [1, 2, 6, 7]},
        {"action": "validate_allowed_values", "column": "rate_code_id", "allowed_values": [1, 2, 3, 4, 5, 6, 99]},
        {"action": "validate_allowed_values", "column": "store_and_fwd_flag", "allowed_values": ["Y", "N"]},
        {"action": "validate_allowed_values", "column": "payment_type", "allowed_values": [0, 1, 2, 3, 4, 5, 6]},
        *[{"action": "validate_non_negative_numbers", "column": column}
        #    **({"expected_invalid_count": 136567} if column == "fare_amount" else {})}
          for column in [
              "passenger_count", "fare_amount", "extra", "mta_tax", "tip_amount",
              "tolls_amount", "improvement_surcharge", "total_amount",
              "congestion_surcharge", "airport_fee", "trip_distance",
          ]]
        # {"action": "validate_null_values", "column": "fare_amount"},
    ],
    "taxi_zones": [
        {"action": "validate_primary_key", "columns": ["location_id"]},
        {"action": "validate_null_values", "column": "borough"},
        {"action": "validate_null_values", "column": "zone"},
        {"action": "validate_duplicates", "columns": "*"},
    ],
    "weather": [
        {"action": "validate_primary_key", "columns": ["year", "month", "day", "hour"]},
        # {"action": "validate_null_values", "column": "observation_datetime"},
        {"action": "validate_duplicates", "columns": "*"},
        {"action": "validate_allowed_values", "column": "month", "allowed_values": list(range(1, 13))},
        {"action": "validate_allowed_values", "column": "hour", "allowed_values": list(range(24))},
        {"action": "validate_allowed_values", "column": "day", "allowed_values": list(range(1, 32))},
        *[{"action": "validate_non_negative_numbers", "column": column} for column in [
            "year", "rhum", "prcp", "snwd", "wdir", "wspd", "wpgt", "pres", "cldc", #"coco",
        ]],
    ],
    "air_quality": [
        # {"action": "validate_null_values", "column": "observation_datetime"},
        {"action": "validate_null_values", "column": "latitude"},
        {"action": "validate_null_values", "column": "longitude"},
        {"action": "validate_null_values", "column": "sample_measurement"},
        {"action": "validate_duplicates", "columns": "*"},
    ],
}

if DATASET not in DATASETS:
    raise ValueError(f"Unknown dataset: {DATASET}. Choose from {sorted(DATASETS)}")

DATASET_CONFIG = DATASETS[DATASET]
SOURCE_FILES = [
    DATA_ROOT / DATASET_CONFIG["directory"] / filename
    for filename in DATASET_CONFIG["files"]
]
RULES = QUALITY_RULES[DATASET]

In [2]:
missing_files = [path for path in SOURCE_FILES if not path.exists()]
if missing_files:
    missing_text = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(f"Missing source files:\n{missing_text}")

frames = []
for source_file in SOURCE_FILES:
    if DATASET_CONFIG["format"] == "parquet":
        frame = pd.read_parquet(source_file)
    else:
        frame = pd.read_csv(source_file)
    frame["source_file"] = source_file.name
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)

# Reproduce the column names and derived fields used by the Spark operations.
if DATASET == "taxi_trips":
    data = data.rename(columns={
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pickup_zone_id",
        "DOLocationID": "dropoff_zone_id",
        "Airport_fee": "airport_fee",
    })
elif DATASET == "taxi_zones":
    data = data.rename(columns={
        "LocationID": "location_id",
        "Borough": "borough",
        "Zone": "zone",
    })
elif DATASET == "air_quality":
    data.columns = [
        column.strip().lower().replace(" ", "_")
        for column in data.columns
    ]
    data["observation_datetime"] = pd.to_datetime(
        data["date_gmt"].astype(str) + " " + data["time_gmt"].astype(str),
        format="mixed",
        errors="coerce",
    )
elif DATASET == "weather":
    data["observation_datetime"] = pd.to_datetime(
        data[["year", "month", "day", "hour"]].rename(columns={
            "year": "year", "month": "month", "day": "day", "hour": "hour",
        }),
        errors="coerce",
    )

print(f"Loaded {len(data):,} rows for {DATASET} from {len(SOURCE_FILES)} file(s).")
print("Columns:", list(data.columns))
display(data.head())
display(data.dtypes.to_frame("dtype"))

Loaded 9,554,778 rows for taxi_trips from 3 file(s).
Columns: ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_fwd_flag', 'pickup_zone_id', 'dropoff_zone_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'source_file']


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0,yellow_tripdata_2024-01.parquet
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,yellow_tripdata_2024-01.parquet
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0,yellow_tripdata_2024-01.parquet
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0,yellow_tripdata_2024-01.parquet
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0,yellow_tripdata_2024-01.parquet


,dtype
vendor_id,int32
pickup_datetime,datetime64[us]
dropoff_datetime,datetime64[us]
passenger_count,float64
trip_distance,float64
rate_code_id,float64
store_and_fwd_flag,str
pickup_zone_id,int32
dropoff_zone_id,int32
payment_type,int64


In [3]:
data.isna().sum()

vendor_id                     0
pickup_datetime               0
dropoff_datetime              0
passenger_count          751962
trip_distance                 0
rate_code_id             751962
store_and_fwd_flag       751962
pickup_zone_id                0
dropoff_zone_id               0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     751962
airport_fee              751962
source_file                   0
dtype: int64

In [4]:
data.shape

(9554778, 20)

## Reproduce configured numeric checks

The notebook reports null, negative, and combined invalid counts for the numeric columns configured for the selected dataset. For `taxi_trips`, it also compares `fare_amount` against the expected Spark count.

In [5]:
def invalid_null_rows(frame: pd.DataFrame, column: str, null_values=None) -> pd.Series:
    mask = frame[column].isna()
    for null_value in null_values or []:
        mask = mask | frame[column].eq(null_value)
    return mask


def invalid_non_negative_rows(frame: pd.DataFrame, column: str) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    return values.lt(0)

# TODO: this needs some fixing!
def duplicate_rows(frame: pd.DataFrame, columns="*") -> pd.DataFrame:
    selected_columns = list(frame.columns) if columns == "*" else list(columns)
    selected_columns = [column for column in selected_columns if column in frame.columns]
    if not selected_columns:
        raise ValueError("No duplicate-check columns exist in the selected dataset.")
    eligible_rows = frame[selected_columns].notna().all(axis=1)
    duplicate_mask = frame.duplicated(
        subset=selected_columns,
        keep=False,
    ) & eligible_rows
    return frame.loc[duplicate_mask]


def primary_key_invalid_rows(frame: pd.DataFrame, columns) -> pd.Series:
    missing = frame[list(columns)].isna().any(axis=1)
    duplicate = frame.duplicated(subset=list(columns), keep=False)
    return missing | duplicate


def evaluate_rule(frame: pd.DataFrame, rule: dict) -> tuple[pd.Series, str]:
    action = rule["action"]
    column = rule.get("column")

    if action == "validate_null_values":
        if column == "*":
            invalid = frame.isna().any(axis=1)
            description = "null values in any column"
        else:
            invalid = invalid_null_rows(frame, column, rule.get("null_values"))
            description = f"null values in {column}"
    elif action == "validate_non_negative_numbers":
        invalid = invalid_non_negative_rows(frame, column)
        description = f"negative values in {column}"
    elif action == "validate_allowed_values":
        invalid = frame[column].notna() & ~frame[column].isin(rule["allowed_values"])
        description = f"non-null values outside allowed set in {column}"
        # invalid = ~frame[column].isin(rule["allowed_values"])
        # description = f"values outside allowed set in {column}"
    elif action == "validate_duplicates":
        duplicate_frame = duplicate_rows(frame, rule["columns"])
        invalid = frame.index.isin(duplicate_frame.index)
        description = f"duplicate rows for {rule['columns']}"
    elif action == "validate_primary_key":
        invalid = primary_key_invalid_rows(frame, rule["columns"])
        description = f"missing or duplicate primary key {rule['columns']}"
    else:
        raise ValueError(f"Unsupported quality rule: {action}")

    return pd.Series(invalid, index=frame.index), description


rule_results = []
invalid_samples = {}
for rule_number, rule in enumerate(RULES, start=1):
    invalid_mask, description = evaluate_rule(data, rule)
    invalid_count = int(invalid_mask.sum())
    expected_count = rule.get("expected_invalid_count")
    rule_results.append({
        "rule": rule_number,
        "action": rule["action"],
        "column(s)": rule.get("column", rule.get("columns", "")),
        "invalid_rows": invalid_count,
        "expected_rows": expected_count,
        "matches_expected": expected_count is None or invalid_count == expected_count,
        "description": description,
    })
    if invalid_count:
        invalid_samples[rule_number] = data.loc[invalid_mask].head(20)

rule_results = pd.DataFrame(rule_results)
display(rule_results)

,rule,action,column(s),invalid_rows,expected_rows,matches_expected,description
0,1,validate_null_values,*,751962,None,True,null values in any column
1,2,validate_duplicates,"[vendor_id, pickup_datetime, dropoff_datetime,...",2,None,True,"duplicate rows for ['vendor_id', 'pickup_datet..."
2,3,validate_allowed_values,vendor_id,0,None,True,non-null values outside allowed set in vendor_id
3,4,validate_allowed_values,rate_code_id,0,None,True,non-null values outside allowed set in rate_co...
4,5,validate_allowed_values,store_and_fwd_flag,0,None,True,non-null values outside allowed set in store_a...
5,6,validate_allowed_values,payment_type,0,None,True,non-null values outside allowed set in payment...
6,7,validate_non_negative_numbers,passenger_count,0,None,True,negative values in passenger_count
7,8,validate_non_negative_numbers,fare_amount,136567,None,True,negative values in fare_amount
8,9,validate_non_negative_numbers,extra,57864,None,True,negative values in extra
9,10,validate_non_negative_numbers,mta_tax,112407,None,True,negative values in mta_tax


In [6]:
duplicate_key = [
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "rate_code_id",
    "store_and_fwd_flag",
    "pickup_zone_id",
    "dropoff_zone_id",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]

# Match the duplicate-check logic: exclude rows with null key values.
eligible = data[duplicate_key].notna().all(axis=1)

grouped_duplicates = (
    data.loc[eligible]
    .groupby(duplicate_key, dropna=False)
    .size()
    .reset_index(name="occurrences")
)

duplicate_groups = grouped_duplicates[grouped_duplicates["occurrences"] > 1]

print(f"Rows checked: {eligible.sum():,}")
print(f"Duplicate groups: {len(duplicate_groups):,}")
print(f"Rows in duplicate groups: {duplicate_groups['occurrences'].sum():,}")
print(f"Unique originals represented: {len(duplicate_groups):,}")
print(
    f"Excess duplicate rows: "
    f"{(duplicate_groups['occurrences'] - 1).sum():,}"
)

display(
    duplicate_groups
    .sort_values("occurrences", ascending=False)
    .head(20)
)

Rows checked: 8,802,816
Duplicate groups: 1
Rows in duplicate groups: 2
Unique originals represented: 1
Excess duplicate rows: 1


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,occurrences
4836228,2,2024-02-08 18:23:50,2024-02-08 18:38:14,1.0,0.81,1.0,N,162,230,1,12.8,2.5,0.5,3.86,0.0,1.0,23.16,2.5,0.0,2


In [7]:
for rule_number, sample in invalid_samples.items():
    print(f"Rule {rule_number} invalid-row sample:")
    display(sample)

Rule 1 invalid-row sample:


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
2824462,2,2024-01-01 00:34:19,2024-01-01 00:51:22,NaN,2.04,NaN,NaN,143,141,0,12.72,0.0,0.5,0.00,0.0,1.0,16.72,NaN,NaN,yellow_tripdata_2024-01.parquet
2824463,1,2024-01-01 00:14:31,2024-01-01 00:19:29,NaN,1.60,NaN,NaN,236,238,0,9.30,1.0,0.5,2.86,0.0,1.0,17.16,NaN,NaN,yellow_tripdata_2024-01.parquet
2824464,1,2024-01-01 00:35:11,2024-01-01 01:13:40,NaN,0.00,NaN,NaN,142,79,0,21.01,0.0,0.5,0.00,0.0,1.0,25.01,NaN,NaN,yellow_tripdata_2024-01.parquet
2824465,1,2024-01-01 00:33:37,2024-01-01 00:50:34,NaN,0.00,NaN,NaN,237,4,0,17.79,0.0,0.5,0.00,0.0,1.0,21.79,NaN,NaN,yellow_tripdata_2024-01.parquet
2824466,1,2024-01-01 00:49:04,2024-01-01 01:01:16,NaN,0.00,NaN,NaN,244,50,0,34.65,0.0,0.5,0.00,0.0,1.0,38.65,NaN,NaN,yellow_tripdata_2024-01.parquet
2824467,2,2024-01-01 00:47:00,2024-01-01 01:11:05,NaN,4.58,NaN,NaN,202,83,0,28.14,0.0,0.5,0.00,0.0,1.0,29.64,NaN,NaN,yellow_tripdata_2024-01.parquet
2824468,2,2024-01-01 00:48:10,2024-01-01 01:00:24,NaN,1.47,NaN,NaN,224,144,0,12.82,0.0,0.5,0.00,0.0,1.0,16.82,NaN,NaN,yellow_tripdata_2024-01.parquet
2824469,1,2024-01-01 00:53:57,2024-01-01 01:09:58,NaN,1.70,NaN,NaN,79,249,0,10.16,0.0,0.5,0.00,0.0,1.0,11.66,NaN,NaN,yellow_tripdata_2024-01.parquet
2824470,2,2024-01-01 00:55:27,2024-01-01 01:38:45,NaN,4.80,NaN,NaN,158,239,0,33.12,0.0,0.5,7.42,0.0,1.0,44.54,NaN,NaN,yellow_tripdata_2024-01.parquet
2824471,2,2024-01-01 00:36:00,2024-01-01 00:43:00,NaN,1.27,NaN,NaN,211,90,0,11.74,0.0,0.5,3.15,0.0,1.0,18.89,NaN,NaN,yellow_tripdata_2024-01.parquet


Rule 2 invalid-row sample:


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
3718872,2,2024-02-08 18:23:50,2024-02-08 18:38:14,1.0,0.81,1.0,N,162,230,1,12.8,2.5,0.5,3.86,0.0,1.0,23.16,2.5,0.0,yellow_tripdata_2024-02.parquet
3718876,2,2024-02-08 18:23:50,2024-02-08 18:38:14,1.0,0.81,1.0,N,162,230,1,12.8,2.5,0.5,3.86,0.0,1.0,23.16,2.5,0.0,yellow_tripdata_2024-02.parquet


Rule 8 invalid-row sample:


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
99,2,2024-01-01 00:18:24,2024-01-01 00:30:39,1.0,2.16,1.0,N,249,232,4,-13.5,-1.0,-0.5,0.0,0.0,-1.0,-18.50,-2.5,0.00,yellow_tripdata_2024-01.parquet
506,2,2024-01-01 00:04:00,2024-01-01 00:04:44,1.0,0.01,5.0,N,63,63,2,-31.5,0.0,0.0,0.0,0.0,-1.0,-34.25,0.0,-1.75,yellow_tripdata_2024-01.parquet
536,2,2024-01-01 00:41:42,2024-01-01 00:46:00,1.0,0.47,1.0,N,249,113,4,-5.8,-1.0,-0.5,0.0,0.0,-1.0,-10.80,-2.5,0.00,yellow_tripdata_2024-01.parquet
552,2,2024-01-01 00:42:02,2024-01-01 01:14:33,1.0,5.48,1.0,N,107,61,2,-33.1,-1.0,-0.5,0.0,0.0,-1.0,-38.10,-2.5,0.00,yellow_tripdata_2024-01.parquet
682,2,2024-01-01 00:24:02,2024-01-01 01:10:32,1.0,8.74,1.0,N,90,225,4,-47.8,-1.0,-0.5,0.0,0.0,-1.0,-52.80,-2.5,0.00,yellow_tripdata_2024-01.parquet
999,2,2024-01-01 00:14:22,2024-01-01 00:21:53,1.0,1.17,1.0,N,170,161,4,-9.3,-1.0,-0.5,0.0,0.0,-1.0,-14.30,-2.5,0.00,yellow_tripdata_2024-01.parquet
1057,2,2024-01-01 00:45:56,2024-01-01 00:56:05,3.0,1.57,1.0,N,79,234,4,-11.4,-1.0,-0.5,0.0,0.0,-1.0,-16.40,-2.5,0.00,yellow_tripdata_2024-01.parquet
1195,2,2024-01-01 00:30:18,2024-01-01 00:57:39,1.0,9.60,1.0,N,79,69,4,-40.1,-1.0,-0.5,0.0,0.0,-1.0,-45.10,-2.5,0.00,yellow_tripdata_2024-01.parquet
1382,2,2024-01-01 00:33:28,2024-01-01 00:33:57,2.0,0.00,1.0,N,162,162,4,-3.0,-1.0,-0.5,0.0,0.0,-1.0,-8.00,-2.5,0.00,yellow_tripdata_2024-01.parquet
1472,2,2024-01-01 00:30:10,2024-01-01 00:40:01,2.0,1.58,1.0,N,249,231,4,-11.4,-1.0,-0.5,0.0,0.0,-1.0,-16.40,-2.5,0.00,yellow_tripdata_2024-01.parquet


... (12 outputs left out)

Rule 15 invalid-row sample:


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
99,2,2024-01-01 00:18:24,2024-01-01 00:30:39,1.0,2.16,1.0,N,249,232,4,-13.5,-1.0,-0.5,0.0,0.0,-1.0,-18.5,-2.5,0.0,yellow_tripdata_2024-01.parquet
536,2,2024-01-01 00:41:42,2024-01-01 00:46:00,1.0,0.47,1.0,N,249,113,4,-5.8,-1.0,-0.5,0.0,0.0,-1.0,-10.8,-2.5,0.0,yellow_tripdata_2024-01.parquet
552,2,2024-01-01 00:42:02,2024-01-01 01:14:33,1.0,5.48,1.0,N,107,61,2,-33.1,-1.0,-0.5,0.0,0.0,-1.0,-38.1,-2.5,0.0,yellow_tripdata_2024-01.parquet
682,2,2024-01-01 00:24:02,2024-01-01 01:10:32,1.0,8.74,1.0,N,90,225,4,-47.8,-1.0,-0.5,0.0,0.0,-1.0,-52.8,-2.5,0.0,yellow_tripdata_2024-01.parquet
999,2,2024-01-01 00:14:22,2024-01-01 00:21:53,1.0,1.17,1.0,N,170,161,4,-9.3,-1.0,-0.5,0.0,0.0,-1.0,-14.3,-2.5,0.0,yellow_tripdata_2024-01.parquet
1057,2,2024-01-01 00:45:56,2024-01-01 00:56:05,3.0,1.57,1.0,N,79,234,4,-11.4,-1.0,-0.5,0.0,0.0,-1.0,-16.4,-2.5,0.0,yellow_tripdata_2024-01.parquet
1195,2,2024-01-01 00:30:18,2024-01-01 00:57:39,1.0,9.60,1.0,N,79,69,4,-40.1,-1.0,-0.5,0.0,0.0,-1.0,-45.1,-2.5,0.0,yellow_tripdata_2024-01.parquet
1382,2,2024-01-01 00:33:28,2024-01-01 00:33:57,2.0,0.00,1.0,N,162,162,4,-3.0,-1.0,-0.5,0.0,0.0,-1.0,-8.0,-2.5,0.0,yellow_tripdata_2024-01.parquet
1472,2,2024-01-01 00:30:10,2024-01-01 00:40:01,2.0,1.58,1.0,N,249,231,4,-11.4,-1.0,-0.5,0.0,0.0,-1.0,-16.4,-2.5,0.0,yellow_tripdata_2024-01.parquet
1712,2,2024-01-01 00:20:38,2024-01-01 00:32:34,1.0,1.89,1.0,N,170,113,4,-12.8,-1.0,-0.5,0.0,0.0,-1.0,-17.8,-2.5,0.0,yellow_tripdata_2024-01.parquet


Rule 16 invalid-row sample:


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_zone_id,dropoff_zone_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
506,2,2024-01-01 00:04:00,2024-01-01 00:04:44,1.0,0.01,5.0,N,63,63,2,-31.5,0.0,0.0,0.0,0.00,-1.0,-34.25,0.0,-1.75,yellow_tripdata_2024-01.parquet
2701,2,2024-01-01 00:48:30,2024-01-01 01:09:02,1.0,15.79,4.0,N,132,15,4,-91.9,-1.0,-0.5,0.0,0.00,-1.0,-96.15,0.0,-1.75,yellow_tripdata_2024-01.parquet
3922,2,2024-01-01 00:46:33,2024-01-01 01:24:05,3.0,11.56,1.0,N,132,61,4,-49.9,-1.0,-0.5,0.0,0.00,-1.0,-54.15,0.0,-1.75,yellow_tripdata_2024-01.parquet
4818,2,2024-01-01 00:19:00,2024-01-01 00:33:56,1.0,7.38,1.0,N,138,256,4,-31.7,-6.0,-0.5,0.0,0.00,-1.0,-43.45,-2.5,-1.75,yellow_tripdata_2024-01.parquet
4911,2,2024-01-01 00:35:57,2024-01-01 01:22:32,1.0,41.20,4.0,N,132,265,2,-201.8,-1.0,-0.5,0.0,-6.94,-1.0,-212.99,0.0,-1.75,yellow_tripdata_2024-01.parquet
5086,2,2024-01-01 00:13:57,2024-01-01 00:19:46,3.0,2.91,1.0,N,132,216,2,-13.5,-1.0,-0.5,0.0,0.00,-1.0,-17.75,0.0,-1.75,yellow_tripdata_2024-01.parquet
8746,2,2024-01-01 01:00:45,2024-01-01 01:00:59,3.0,0.00,3.0,N,132,132,4,-23.0,-1.0,0.0,0.0,0.00,-1.0,-26.75,0.0,-1.75,yellow_tripdata_2024-01.parquet
10814,2,2024-01-01 01:12:33,2024-01-01 01:54:28,4.0,19.59,3.0,N,132,263,3,-100.7,-1.0,0.0,0.0,-23.94,-1.0,-128.39,0.0,-1.75,yellow_tripdata_2024-01.parquet
15919,2,2024-01-01 02:10:18,2024-01-01 02:10:27,1.0,0.02,1.0,N,132,132,3,-3.0,-1.0,-0.5,0.0,0.00,-1.0,-7.25,0.0,-1.75,yellow_tripdata_2024-01.parquet
15976,2,2024-01-01 02:58:48,2024-01-01 03:01:33,1.0,1.49,1.0,N,132,132,4,-7.9,-1.0,-0.5,0.0,0.00,-1.0,-12.15,0.0,-1.75,yellow_tripdata_2024-01.parquet


## Inspect invalid rows

For taxi trips, show sample invalid fare rows and their distribution by source file. For other datasets, the numeric summary cell is the main result.

In [52]:
rule_results[[
    "rule",
    "action",
    "column(s)",
    "invalid_rows",
    "expected_rows",
    "matches_expected",
]]

,rule,action,column(s),invalid_rows,expected_rows,matches_expected
0,1,validate_null_values,*,751962,None,True
1,2,validate_duplicates,"[vendor_id, pickup_datetime, dropoff_datetime,...",2,None,True
2,3,validate_allowed_values,vendor_id,0,None,True
3,4,validate_allowed_values,rate_code_id,0,None,True
4,5,validate_allowed_values,store_and_fwd_flag,0,None,True
5,6,validate_allowed_values,payment_type,0,None,True
6,7,validate_non_negative_numbers,passenger_count,0,None,True
7,8,validate_non_negative_numbers,fare_amount,136567,None,True
8,9,validate_non_negative_numbers,extra,57864,None,True
9,10,validate_non_negative_numbers,mta_tax,112407,None,True


## Check Duplicate Records

A duplicate is a row with the same values in every selected column. The notebook checks complete rows and then applies a dataset-specific key: taxi trip fields, weather timestamp fields, taxi-zone ID, or air-quality identity fields.

In [53]:
duplicate_results = rule_results[
    rule_results["action"].isin(["validate_duplicates", "validate_primary_key"])
]
display(duplicate_results)

for _, result in duplicate_results.iterrows():
    print(
        f"Rule {result['rule']}: {result['description']} -> "
        f"{result['invalid_rows']:,} invalid rows"
    )

,rule,action,column(s),invalid_rows,expected_rows,matches_expected,description
1,2,validate_duplicates,"[vendor_id, pickup_datetime, dropoff_datetime,...",2,None,True,"duplicate rows for ['vendor_id', 'pickup_datet..."


Rule 2: duplicate rows for ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_fwd_flag', 'pickup_zone_id', 'dropoff_zone_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee'] -> 2 invalid rows
